In [ ]:
## link to investments. prioritse source that has investment & capacity. 

## if a date is missing, don't process further but raise an error

## probably should flag rows we include rather than dedup

In [21]:
import pandas as pd
from collections import defaultdict

import sys; sys.path.append("..")
from src.vote_helpers import fill_single_product_lv2, parse_capacity_value
from mongo_client import mongo_client, capacities_collection
from src.config import GRPD_PROJECTS_FILTER

try:
    mongo_client.admin.command("ping")
    print("✅ Connected to MongoDB Atlas!")
except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}")
    raise

df = pd.read_excel(GRPD_PROJECTS_FILTER)

# parse dates; push NaT to the end for sorting
df["date"] = pd.to_datetime(df["date"], format='%Y-%m', errors="coerce")
df["_sort_date"] = df["date"].fillna(pd.Timestamp.max)

# define categorical status order of priority
STATUS_ORDER = ["operational", "under construction", "announced", "unclear"] # rumour (clearer than unclear)
df["status"] = pd.Categorical(df["status"], categories=STATUS_ORDER, ordered=True)
status_rank = df["status"].cat.codes
status_rank = status_rank.where(status_rank >= 0, len(STATUS_ORDER))  # unknown -> worst

# normalise capacities
df["capacity_normalized"] = df["capacity_normalized"].apply(parse_capacity_value)
print(f"Length of initial dataframe: {len(df)}")

# keys, we are adding identical capacity values as a second key
keys = ["cluster_id", "capacity_normalized"]

# FIRST deduplication
# cases with identical capacity value, status and phase. 
# we keep the earliest date

df1 = (
    df.sort_values(
        ["cluster_id", "date"],
        ascending=[True, True],
        na_position="last"   # puts NaT after real dates
    )
    .drop_duplicates(["cluster_id", "capacity_normalized", "status", "phase"], keep="first")
)
print(f"📅 Length with only earliest dates for identical capacity | status | phase: {len(df1)}")

# for each identical capacity value. 
# calculate earliest_announce_date 

## we define this as the earliest data with status == announced (& type == greenfi)

# a quick solution for returning earliest data per cluster_id and capacity_normalized
earliest_dates = (
    df.groupby(["cluster_id", "capacity_normalized"])["date"]
      .min()          # skipna=True by default
      .reset_index()
)

# 3) Sort + drop_duplicates to keep the single best row per group

dedup = (
    df.assign(_status_rank=status_rank)
      .sort_values(keys + ["_sort_date", "_status_rank"])
      .drop_duplicates(keys, keep="first")
      #.drop(columns=["_sort_date", "_status_rank"])
)

✅ Connected to MongoDB Atlas!
Length of initial dataframe: 347
📅 Length with only earliest dates for identical capacity | status | phase: 231


In [22]:
df1

,inst_canon,city_key,adm1,adm2,iso2,bbox,cluster_id,product_lv1,product_lv2,product,capacity_normalized,capacity_metric_normalized,status,phase,date,article_id,_sort_date
0,morrow batteries,arendal,Agder,Arendal,NO,"{'east': 8.815542328183053, 'south': 58.439025...",2,battery,cell,battery cells,32.00,gigawatt hour,announced,greenfield,2020-12-01,67f92144f431fddd61d55a86,2020-12-01 00:00:00.000000000
2,morrow batteries,arendal,Agder,Arendal,NO,"{'east': 8.815542328183053, 'south': 58.439025...",2,battery,cell,battery cells,42.00,gigawatt hour,announced,greenfield,2021-05-01,67f92171f431fddd61d55b1a,2021-05-01 00:00:00.000000000
4,morrow batteries,arendal,Agder,Arendal,NO,"{'east': 8.815542328183053, 'south': 58.439025...",2,battery,cell,battery cells,43.00,gigawatt hour,under construction,greenfield,2022-11-01,67f92232f431fddd61d55d99,2022-11-01 00:00:00.000000000
6,morrow batteries,arendal,Agder,Arendal,NO,"{'east': 8.815542328183053, 'south': 58.439025...",2,battery,NaN,LFP batteries,1.00,gigawatt,operational,greenfield,2024-08-01,67f92304f431fddd61d56050,2024-08-01 00:00:00.000000000
7,stellantis catl,zaragoza,Aragon,Saragossa,ES,"{'east': -0.7344216175641901, 'south': 41.5494...",21,battery,cell,LFP battery cells,50.00,gigawatt hour,announced,greenfield,NaT,67f9231ef431fddd61d560a4,2262-04-11 23:47:16.854775807
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
338,novo energy,gothenburg,Västra Götaland,Gothenburg Municipality,SE,"{'east': 12.180168053211563, 'south': 57.46628...",509,battery,cell,battery cell,50.00,gigawatt hour,under construction,greenfield,2023-09-01,67f92295f431fddd61d55ee3,2023-09-01 00:00:00.000000000
342,volvo northvolt,gothenburg,Västra Götaland,Gothenburg Municipality,SE,"{'east': 12.180168053211563, 'south': 57.46628...",511,battery,cell,battery cells,50.00,gigawatt hour,announced,greenfield,2022-02-01,67f921a4f431fddd61d55bc8,2022-02-01 00:00:00.000000000
344,volvo northvolt,gothenburg,Västra Götaland,Gothenburg Municipality,SE,"{'east': 12.180168053211563, 'south': 57.46628...",511,battery,cell,battery cells,50.00,gigawatt hour,under construction,greenfield,2024-02-01,67f52c98981040986eab79af,2024-02-01 00:00:00.000000000
345,battrion,dübendorf,Zurich,Bezirk Uster,CH,"{'east': 8.651844002323093, 'south': 47.384252...",529,battery,NaN,electrodes,0.02,gigawatt hour,operational,greenfield,2021-04-01,67f92175f431fddd61d55b28,2021-04-01 00:00:00.000000000


In [ ]:
df2 = (
    df1.sort_values(
        ["cluster_id", "date"],
        ascending=[True, True],
        na_position="last"   
    )
    .drop_duplicates(["cluster_id", "status", "phase"], keep="last")
)

In [19]:
df1[["cluster_id", "capacity_normalized", "status", "phase", "date"]].head(10)

,cluster_id,capacity_normalized,status,phase,date
0,2,32.00,announced,greenfield,2020-12-01
2,2,42.00,announced,greenfield,2021-05-01
4,2,43.00,under construction,greenfield,2022-11-01
6,2,1.00,operational,greenfield,2024-08-01
7,21,50.00,announced,greenfield,NaT
8,31,0.15,announced,greenfield,2023-01-01
9,32,0.10,under construction,greenfield,2022-05-01
10,32,1.30,under construction,greenfield,2024-07-01
11,33,3000.00,operational,greenfield,2025-05-01
12,36,1.15,announced,expansion,2023-07-01


In [16]:
df1[["cluster_id", "capacity_normalized", "status", "phase", "date"]].head(30)

,cluster_id,capacity_normalized,status,phase,date
6,2,1.000,operational,greenfield,2024-08-01
4,2,43.000,under construction,greenfield,2022-11-01
0,2,32.000,announced,greenfield,2020-12-01
2,2,42.000,announced,greenfield,2021-05-01
7,21,50.000,announced,greenfield,NaT
8,31,0.150,announced,greenfield,2023-01-01
9,32,0.100,under construction,greenfield,2022-05-01
10,32,1.300,under construction,greenfield,2024-07-01
11,33,3000.000,operational,greenfield,2025-05-01
14,36,1.300,under construction,greenfield,2024-05-01


In [6]:
df[["cluster_id", "capacity_normalized", "status", "phase", "date"]].head(30)

,cluster_id,capacity_normalized,status,phase,date,_sort_date
0,2,32.000,announced,greenfield,2020-12-01,2020-12-01 00:00:00.000000000
1,2,32.000,announced,greenfield,2020-12-01,2020-12-01 00:00:00.000000000
2,2,42.000,announced,greenfield,2021-05-01,2021-05-01 00:00:00.000000000
3,2,42.000,announced,greenfield,2021-10-01,2021-10-01 00:00:00.000000000
4,2,43.000,under construction,greenfield,2022-11-01,2022-11-01 00:00:00.000000000
5,2,32.000,announced,greenfield,2023-01-01,2023-01-01 00:00:00.000000000
6,2,1.000,operational,greenfield,2024-08-01,2024-08-01 00:00:00.000000000
7,21,50.000,announced,greenfield,NaT,2262-04-11 23:47:16.854775807
8,31,0.150,announced,greenfield,2023-01-01,2023-01-01 00:00:00.000000000
9,32,0.100,under construction,greenfield,2022-05-01,2022-05-01 00:00:00.000000000


In [59]:
strongest_status

NameError: name 'strongest_status' is not defined

In [58]:
earliest_dates

,cluster_id,capacity_normalized,date
0,2,1.00,2024-08-01
1,2,32.00,2020-12-01
2,2,42.00,2021-05-01
3,2,43.00,2022-11-01
4,21,50.00,NaT
...,...,...,...
163,508,50.00,2022-02-01
164,509,50.00,2023-07-01
165,511,50.00,2022-02-01
166,529,0.02,2021-04-01


In [43]:
df["date"]

0     2020-12-01
1     2020-12-01
2     2021-05-01
3     2021-10-01
4     2022-11-01
         ...    
342   2022-02-01
343   2022-04-01
344   2024-02-01
345   2021-04-01
346   2022-04-01
Name: date, Length: 347, dtype: datetime64[ns]

In [ ]:
# fill product_lv2 across multiple NA rows
#df = df.groupby("cluster_id").apply(fill_single_product_lv2).reset_index(drop=True)

# normalise capacity values

#print(len(df))
#print(len(df[(df["phase"] == "unclear")]))
#print(len(df[df["status"] == "unclear"]))
#print(len(df[df["date"].isna()]))

# drop all cases where phase | status are unclear

#df = df[(df["phase"] != "unclear") & (df["status"] != "unclear")].copy()
#df = df[~df["date"].isna()].copy()

#print(len(df)) 

In [25]:
df[df["cluster_id"] == 2]

,inst_canon,city_key,adm1,adm2,iso2,bbox,cluster_id,product_lv1,product_lv2,product,capacity_normalized,capacity_metric_normalized,status,phase,date,article_id
0,morrow batteries,arendal,Agder,Arendal,NO,"{'east': 8.815542328183053, 'south': 58.439025...",2,battery,cell,battery cells,32.0,gigawatt hour,announced,greenfield,NaT,67f92144f431fddd61d55a86
1,morrow batteries,arendal,Agder,Arendal,NO,"{'east': 8.815542328183053, 'south': 58.439025...",2,battery,cell,lithium-sulfur battery cells,32.0,gigawatt hour,announced,greenfield,NaT,67f92144f431fddd61d55a86
2,morrow batteries,arendal,Agder,Arendal,NO,"{'east': 8.815542328183053, 'south': 58.439025...",2,battery,cell,battery cells,42.0,gigawatt hour,announced,greenfield,NaT,67f92171f431fddd61d55b1a
3,morrow batteries,arendal,Agder,Arendal,NO,"{'east': 8.815542328183053, 'south': 58.439025...",2,battery,cell,battery cells,42.0,gigawatt hour,announced,greenfield,NaT,67f921bbf431fddd61d55c13
4,morrow batteries,arendal,Agder,Arendal,NO,"{'east': 8.815542328183053, 'south': 58.439025...",2,battery,cell,batteries,32.0,gigawatt hour,announced,greenfield,NaT,67f92217f431fddd61d55d3d
5,morrow batteries,arendal,Agder,Arendal,NO,"{'east': 8.815542328183053, 'south': 58.439025...",2,battery,cell,battery cells,43.0,gigawatt hour,under construction,greenfield,NaT,67f92232f431fddd61d55d99
6,morrow batteries,arendal,Agder,Arendal,NO,"{'east': 8.815542328183053, 'south': 58.439025...",2,battery,cell,LFP batteries,1.0,gigawatt,operational,greenfield,NaT,67f92304f431fddd61d56050


In [ ]:
import logging

# Remove all handlers associated with the root logger object (to avoid duplicate logs in Jupyter)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Set up logging to file and console
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("votes_log.log", mode='w'),  # opening in write mode (overwriting each time)
        logging.StreamHandler()  # This keeps output in the notebook as well
    ]
)

# 🧹 Clear the capacities collection before inserting new data
capacities_collection.delete_many({})

# Example usage
logging.info("INITIATING LOGGING")
unique_clusters = df["cluster_id"].unique().tolist()
logging.info(f"Votes for {len(unique_clusters)} clusters.")
status_counts = defaultdict(int)

for cluster in unique_clusters:

    logging.info(f"Assessing votes for cluster_id: {cluster}")

    df_cluster = df[df["cluster_id"] == cluster].copy()
    unique_statuses = df_cluster["status"].unique()
    unique_phases = df_cluster["phase"].unique()

    #owner = df_cluster["inst_canon"].unique()[0]
    #iso2 = df_cluster["iso2"].unique()[0]
    #adm1 = df_cluster["adm1"].unique()[0]
    #product_lv1 = df_cluster["product_lv1"].unique()[0]

    # Define all possible product_lv2 values
    #product_keys = ["eam", "cell", "module_pack", "recycling"]

    # Set all to False initially
    #product_lv2_dict = {key: False for key in product_keys}

    # Get unique product_lv2 values for this cluster
    #present_products = df_cluster["product_lv2"].dropna().unique().tolist()

    # # Update dict: set to True if present
    # for key in product_keys:
    #     if key in present_products:
    #         product_lv2_dict[key] = True

    if "announced" in unique_statuses and len(unique_statuses) <2:
        logging.info("⚪️ ANNOUNCED case")
        status_counts["announced"] += 1

        # only announcement so we ignore expansions 
        df_canon = df_cluster[df_cluster["phase"] != "expansion"]
        # if there are no greenfield (non-expansion) phases, we drop
        if len(df_canon) < 1:
            logging.info("🔴 ERROR - no greenfield phase noted at facility, only expansions. DROPPED.")
            status_counts["announced-dropped-only-expansions"] += 1
            continue

        dt_announce = df_canon["date"].min()
        phase = "greenfield"

        # find row with the latest date & take capacity
        row_latest = df_canon.loc[df_canon["date"].idxmax()]
        capacity_latest = row_latest["capacity_normalized"]
        
        dt_construct = None
        dt_operation = None

    elif "under construction" in unique_statuses and "operational" not in unique_statuses:
        logging.info("🔵 UNDER CONSTRUCTION case")
        status_counts["under-construction"] += 1

        # only under construction so we ignore expansions 
        df_canon = df_cluster[df_cluster["phase"] != "expansion"]
        if len(df_canon) < 1:
            logging.info("🔴 ERROR - no greenfield phase noted at facility, only expansions. DROPPED.")
            status_counts["under-construction-dropped-only-expansions"] += 1
            continue

        dt_announce = df_canon["date"].min()
        status = "under construction"
        phase = "greenfield"

        # Find row with the latest date & take capacity (for under construction cases)
        df_construct = df_canon[df_canon["status"] == "under construction"].copy()
        dt_construct = df_construct["date"].min()
        row_latest_construction = df_construct.loc[df_construct["date"].idxmax()]
        capacity_latest = row_latest_construction["capacity_normalized"]
        
        dt_operation = None

    elif "operational" in unique_statuses:
        logging.info("🟢 OPERATIONAL case")
        status_counts["operational"] += 1

        # cases where there are no expansions
        if "expansion" not in unique_phases:
            logging.info("No expansions sub-case")
            status_counts["operational-no-expansions"] += 1

            # earliest date taken for dt_announce
            dt_announce = df_cluster["date"].min()    #the earliest date mentioned
            status = "operational"
            phase = "greenfield"

            # earliest date where status == under construction taken for dt_construct
            df_construct = df_cluster[df_cluster["status"] == "under construction"].copy()
            dt_construct = df_construct["date"].min()

            # earliest date where status == operational taken for dt_operation
            df_operation = df_cluster[df_cluster["status"] == "operational"].copy()
            dt_operation = df_operation["date"].min()

            # take operational capacity from most recent article
            row_latest_operation = df_operation.loc[df_operation["date"].idxmax()]
            capacity_latest = row_latest_operation["capacity_normalized"]

        if "expansion" in unique_phases:
            logging.info("With expansions sub-case")
            status_counts["operational-with-expansions"] += 1

    else:
        logging.info("Group not assigned")
        status_counts["unassigned"] += 1

        # only consider expansions AFTER the operational Greenfield row. 

    if dt_announce is not None:
    # final dictionary to insert to Mongo
        mongo_entry = {
            "owner": owner,
            "iso2": iso2,
            "adm1": adm1,
            "product_lv1": product_lv1,
            "product_lv2": product_lv2_dict,
            "dt_announce": dt_announce.strftime("%Y-%m") if pd.notnull(dt_announce) else None,
            "status": status,
            "phase": phase,
            "capacity": capacity_latest,
            "dt_construct": dt_construct.strftime("%Y-%m") if pd.notnull(dt_construct) else None,
            "dt_operation": dt_operation.strftime("%Y-%m") if pd.notnull(dt_operation) else None
        }

        capacities_collection.insert_one(mongo_entry)
        logging.info("✅ Inserted announced project into MongoDB.")

logging.info("📊 Final Status Counts:")
for status, count in status_counts.items():
    logging.info(f"{status}: {count}")

2025-07-17 18:15:55,384 - INFO - INITIATING LOGGING
2025-07-17 18:15:55,384 - INFO - Votes for 65 clusters.
2025-07-17 18:15:55,385 - INFO - Assessing votes for cluster_id: 35
2025-07-17 18:15:55,386 - INFO - 🔵 UNDER CONSTRUCTION case
2025-07-17 18:15:55,462 - INFO - ✅ Inserted announced project into MongoDB.
2025-07-17 18:15:55,463 - INFO - Assessing votes for cluster_id: 37
2025-07-17 18:15:55,464 - INFO - 🔵 UNDER CONSTRUCTION case
2025-07-17 18:15:55,540 - INFO - ✅ Inserted announced project into MongoDB.
2025-07-17 18:15:55,540 - INFO - Assessing votes for cluster_id: 39
2025-07-17 18:15:55,542 - INFO - 🔵 UNDER CONSTRUCTION case


{'owner': 'porsche', 'iso2': 'DE', 'adm1': 'Baden-Wurttemberg', 'product_lv1': 'battery', 'product_lv2': {'eam': False, 'cell': True, 'module_pack': False, 'recycling': False}, 'dt_announce': '2024-05', 'status': 'under construction', 'phase': 'greenfield', 'capacity': np.float64(1.3), 'dt_construct': '2024-05', 'dt_operation': None}
{'owner': 'valmet automotive', 'iso2': 'DE', 'adm1': 'Baden-Wurttemberg', 'product_lv1': 'battery', 'product_lv2': {'eam': False, 'cell': False, 'module_pack': False, 'recycling': False}, 'dt_announce': '2021-02', 'status': 'under construction', 'phase': 'greenfield', 'capacity': np.float64(nan), 'dt_construct': '2021-02', 'dt_operation': None}
{'owner': 'bmw', 'iso2': 'DE', 'adm1': 'Bavaria', 'product_lv1': 'battery', 'product_lv2': {'eam': False, 'cell': True, 'module_pack': True, 'recycling': False}, 'dt_announce': '2024-04', 'status': 'under construction', 'phase': 'greenfield', 'capacity': np.float64(nan), 'dt_construct': '2024-04', 'dt_operation': No

2025-07-17 18:15:55,610 - INFO - ✅ Inserted announced project into MongoDB.
2025-07-17 18:15:55,611 - INFO - Assessing votes for cluster_id: 41
2025-07-17 18:15:55,613 - INFO - 🟢 OPERATIONAL case
2025-07-17 18:15:55,613 - INFO - No expansions sub-case
2025-07-17 18:15:55,681 - INFO - ✅ Inserted announced project into MongoDB.
2025-07-17 18:15:55,681 - INFO - Assessing votes for cluster_id: 42
2025-07-17 18:15:55,684 - INFO - 🟢 OPERATIONAL case
2025-07-17 18:15:55,684 - INFO - With expansions sub-case
2025-07-17 18:15:55,770 - INFO - ✅ Inserted announced project into MongoDB.
2025-07-17 18:15:55,771 - INFO - Assessing votes for cluster_id: 43
2025-07-17 18:15:55,774 - INFO - ⚪️ ANNOUNCED case


{'owner': 'fenecon', 'iso2': 'DE', 'adm1': 'Bavaria', 'product_lv1': 'battery', 'product_lv2': {'eam': False, 'cell': False, 'module_pack': False, 'recycling': False}, 'dt_announce': '2024-04', 'status': 'operational', 'phase': 'greenfield', 'capacity': np.float64(500.0), 'dt_construct': None, 'dt_operation': '2024-04'}
{'owner': 'kion battery systems', 'iso2': 'DE', 'adm1': 'Bavaria', 'product_lv1': 'battery', 'product_lv2': {'eam': False, 'cell': False, 'module_pack': False, 'recycling': False}, 'dt_announce': '2024-04', 'status': 'operational', 'phase': 'greenfield', 'capacity': np.float64(500.0), 'dt_construct': None, 'dt_operation': '2024-04'}
{'owner': 'man', 'iso2': 'DE', 'adm1': 'Bavaria', 'product_lv1': 'battery', 'product_lv2': {'eam': False, 'cell': False, 'module_pack': True, 'recycling': False}, 'dt_announce': '2024-11', 'status': 'operational', 'phase': 'greenfield', 'capacity': np.float64(50000.0), 'dt_construct': None, 'dt_operation': None}


2025-07-17 18:15:55,851 - INFO - ✅ Inserted announced project into MongoDB.
2025-07-17 18:15:55,852 - INFO - Assessing votes for cluster_id: 49
2025-07-17 18:15:55,854 - INFO - 🟢 OPERATIONAL case
2025-07-17 18:15:55,856 - INFO - With expansions sub-case
2025-07-17 18:15:55,929 - INFO - ✅ Inserted announced project into MongoDB.
2025-07-17 18:15:55,930 - INFO - Assessing votes for cluster_id: 50
2025-07-17 18:15:55,932 - INFO - 🔵 UNDER CONSTRUCTION case
2025-07-17 18:15:55,999 - INFO - ✅ Inserted announced project into MongoDB.
2025-07-17 18:15:55,999 - INFO - Assessing votes for cluster_id: 51
2025-07-17 18:15:56,001 - INFO - ⚪️ ANNOUNCED case


...

2025-07-17 18:15:59,510 - INFO - ✅ Inserted announced project into MongoDB.
2025-07-17 18:15:59,511 - INFO - Assessing votes for cluster_id: 486
2025-07-17 18:15:59,514 - INFO - 🔵 UNDER CONSTRUCTION case
2025-07-17 18:15:59,591 - INFO - ✅ Inserted announced project into MongoDB.
2025-07-17 18:15:59,592 - INFO - Assessing votes for cluster_id: 491
2025-07-17 18:15:59,594 - INFO - 🟢 OPERATIONAL case
2025-07-17 18:15:59,598 - INFO - With expansions sub-case
2025-07-17 18:15:59,671 - INFO - ✅ Inserted announced project into MongoDB.
2025-07-17 18:15:59,671 - INFO - Assessing votes for cluster_id: 495
2025-07-17 18:15:59,672 - INFO - 🔵 UNDER CONSTRUCTION case


{'owner': 'volkswagen', 'iso2': 'ES', 'adm1': 'Valencia', 'product_lv1': 'battery', 'product_lv2': {'eam': False, 'cell': True, 'module_pack': False, 'recycling': False}, 'dt_announce': '2022-02', 'status': 'under construction', 'phase': 'greenfield', 'capacity': np.float64(40.0), 'dt_construct': '2024-10', 'dt_operation': None}
{'owner': 'northvolt', 'iso2': 'SE', 'adm1': 'Västerbotten', 'product_lv1': 'battery', 'product_lv2': {'eam': False, 'cell': True, 'module_pack': False, 'recycling': False}, 'dt_announce': '2022-02', 'status': 'under construction', 'phase': 'greenfield', 'capacity': np.float64(40.0), 'dt_construct': '2024-10', 'dt_operation': None}
{'owner': 'northvolt volvo', 'iso2': 'SE', 'adm1': 'Västra Götaland', 'product_lv1': 'battery', 'product_lv2': {'eam': False, 'cell': True, 'module_pack': False, 'recycling': False}, 'dt_announce': '2022-02', 'status': 'under construction', 'phase': 'greenfield', 'capacity': np.float64(50.0), 'dt_construct': '2022-02', 'dt_operation'

2025-07-17 18:15:59,750 - INFO - ✅ Inserted announced project into MongoDB.
2025-07-17 18:15:59,751 - INFO - Assessing votes for cluster_id: 496
2025-07-17 18:15:59,752 - INFO - 🔵 UNDER CONSTRUCTION case
2025-07-17 18:15:59,819 - INFO - ✅ Inserted announced project into MongoDB.
2025-07-17 18:15:59,820 - INFO - Assessing votes for cluster_id: 498
2025-07-17 18:15:59,821 - INFO - 🔵 UNDER CONSTRUCTION case
2025-07-17 18:15:59,889 - INFO - ✅ Inserted announced project into MongoDB.
2025-07-17 18:15:59,889 - INFO - Assessing votes for cluster_id: 515
2025-07-17 18:15:59,891 - INFO - 🟢 OPERATIONAL case
2025-07-17 18:15:59,892 - INFO - No expansions sub-case


{'owner': 'novo energy', 'iso2': 'SE', 'adm1': 'Västra Götaland', 'product_lv1': 'battery', 'product_lv2': {'eam': False, 'cell': True, 'module_pack': False, 'recycling': False}, 'dt_announce': '2023-07', 'status': 'under construction', 'phase': 'greenfield', 'capacity': np.float64(50.0), 'dt_construct': '2024-10', 'dt_operation': None}
{'owner': 'volvo northvolt', 'iso2': 'SE', 'adm1': 'Västra Götaland', 'product_lv1': 'battery', 'product_lv2': {'eam': False, 'cell': True, 'module_pack': False, 'recycling': False}, 'dt_announce': '2021-06', 'status': 'under construction', 'phase': 'greenfield', 'capacity': np.float64(50.0), 'dt_construct': '2024-02', 'dt_operation': None}
{'owner': 'battrion', 'iso2': 'CH', 'adm1': 'Zurich', 'product_lv1': 'battery', 'product_lv2': {'eam': False, 'cell': False, 'module_pack': False, 'recycling': False}, 'dt_announce': '2021-04', 'status': 'operational', 'phase': 'greenfield', 'capacity': np.float64(0.02), 'dt_construct': None, 'dt_operation': '2021-04

2025-07-17 18:15:59,964 - INFO - ✅ Inserted announced project into MongoDB.
2025-07-17 18:15:59,965 - INFO - 📊 Final Status Counts:
2025-07-17 18:15:59,965 - INFO - under-construction: 23
2025-07-17 18:15:59,966 - INFO - operational: 15
2025-07-17 18:15:59,966 - INFO - operational-no-expansions: 6
2025-07-17 18:15:59,966 - INFO - operational-with-expansions: 9
2025-07-17 18:15:59,967 - INFO - announced: 27
2025-07-17 18:15:59,968 - INFO - announced-dropped-only-expansions: 6
2025-07-17 18:15:59,968 - INFO - under-construction-dropped-only-expansions: 1


In [9]:
df[df["cluster_id"] == 49]

,inst_canon,city_key,adm1,adm2,iso2,cluster_id,product_lv1,product_lv2,product,capacity_normalized,capacity_metric_normalized,status,phase,date,article_id
28,microvast gmbh,ludwigsfelde,Brandenburg,NaN,DE,49,battery,module_pack,battery modules,1.5,GWh,operational,greenfield,2021-02-01,67f92164f431fddd61d55af0
29,microvast gmbh,ludwigsfelde,Brandenburg,NaN,DE,49,battery,module_pack,battery modules,6.0,GWh,announced,expansion,2021-02-01,67f92164f431fddd61d55af0


In [14]:
df[df["cluster_id"] == 312]

,inst_canon,city_key,adm1,adm2,iso2,cluster_id,product_lv1,product_lv2,product,capacity_normalized,capacity_metric_normalized,status,phase,date,article_id
133,volkswagen,salzgitter,Lower Saxony,NaN,DE,312,battery,cell,battery cells,12.0,GWh,operational,greenfield,2019-05-01,67f92108f431fddd61d559bc
134,volkswagen,salzgitter,Lower Saxony,NaN,DE,312,battery,cell,battery cells,30.0,GWh,announced,expansion,2019-05-01,67f92108f431fddd61d559bc
135,volkswagen,salzgitter,Lower Saxony,NaN,DE,312,battery,cell,battery cells,16.0,GWh,announced,expansion,2019-09-01,67f92112f431fddd61d559dd
136,volkswagen,salzgitter,Lower Saxony,NaN,DE,312,battery,cell,solid-state lithium-metal batteries,1.0,GWh,announced,greenfield,2021-05-01,67f92163f431fddd61d55aed
137,volkswagen,salzgitter,Lower Saxony,NaN,DE,312,battery,cell,solid-state lithium-metal batteries,21.0,GWh,announced,expansion,2021-05-01,67f92163f431fddd61d55aed
138,volkswagen,salzgitter,Lower Saxony,NaN,DE,312,battery,cell,unit cells,20.0,GWh,announced,greenfield,2021-12-01,67f921a2f431fddd61d55bc1
139,volkswagen,salzgitter,Lower Saxony,NaN,DE,312,battery,cell,unit cells,40.0,GWh,announced,expansion,2021-12-01,67f921a2f431fddd61d55bc1
142,volkswagen,salzgitter,Lower Saxony,NaN,DE,312,battery,cell,battery cells,40.0,GWh,announced,greenfield,2023-10-01,67f92299f431fddd61d55ef2
144,volkswagen,salzgitter,Lower Saxony,NaN,DE,312,battery,cell,unified cell,20.0,GWh,operational,greenfield,2024-09-01,67f92324f431fddd61d560b9
145,volkswagen,salzgitter,Lower Saxony,NaN,DE,312,battery,cell,battery cells,40.0,GWh,under construction,greenfield,2024-12-01,67f92310f431fddd61d56077
